In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import make_scorer, f1_score, accuracy_score, precision_score, recall_score, roc_auc_score
from sklearn.ensemble import RandomForestClassifier

from joblib import dump

SEED = 42
np.random.seed(SEED)
sns.set_style("whitegrid")

# Load the UCI dataset (adjust path as needed)
DATA_PATH = "online_shoppers_intention.csv"
df = pd.read_csv(DATA_PATH)

# Ensure binary label 0/1
TARGET_COL = "Revenue"
df[TARGET_COL] = df[TARGET_COL].astype(int)

# Holdout once (20%) – do NOT touch until the very end
train_df, holdout_df = train_test_split(
    df, test_size=0.20, random_state=SEED, stratify=df[TARGET_COL]
)
X_train, y_train = train_df.drop(columns=[TARGET_COL]), train_df[TARGET_COL].values
X_holdout, y_holdout = holdout_df.drop(columns=[TARGET_COL]), holdout_df[TARGET_COL].values

print(train_df.shape, holdout_df.shape, y_train.mean().round(3), y_holdout.mean().round(3))


In [ ]:
# Either load your canonical engineered dataset (features + Revenue)...
df = pd.read_csv("../data/train_data_with_engineered.csv")
y = df["Revenue"].astype(int)
X_full = df.drop(columns=["Revenue"])

# ...and the selected feature list from your feature selection step
with open("../results/feature_set_plan.json", "r") as f:
    plan = json.load(f)

best_name  = plan["best_overall"]["name"]       # e.g., "num_pruned_engineered_plus_temporal"
best_cols  = plan["best_overall"]["columns"]    # exact list of columns
X = X_full[[c for c in best_cols if c in X_full.columns]].copy()

print(f"Using feature set: {best_name} | {X.shape}")


In [ ]:
# Heuristic detection; override CAT_COLS manually if needed
CAT_COLS = [c for c in X.columns if X[c].dtype == "object" or str(X[c].dtype).startswith(("category","bool"))]
NUM_COLS = [c for c in X.columns if c not in CAT_COLS]

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), NUM_COLS),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT_COLS),
], remainder="drop")

print(f"NUM: {len(NUM_COLS)} | CAT: {len(CAT_COLS)}")


In [ ]:
def run_search(name, estimator, param_grid, search="grid", n_iter=20, refit=PRIMARY):
    """
    Runs a CV search with a preprocessing pipeline.
    - search="grid"  -> GridSearchCV(param_grid=...)
    - search="random"-> RandomizedSearchCV(param_distributions=..., n_iter=n_iter)
    Returns: (best_estimator_pipeline, result_row_dict)
    """
    pipe = Pipeline([("prep", preprocessor), ("clf", estimator)])
    grid = {f"clf__{k}": v for k, v in param_grid.items()}

    if search == "grid":
        searcher = GridSearchCV(
            estimator=pipe,
            param_grid=grid,
            cv=cv,
            scoring=PRIMARY,
            n_jobs=-1,
            verbose=1,
            refit=True,
        )
    else:  # "random"
        searcher = RandomizedSearchCV(
            estimator=pipe,
            param_distributions=grid,
            n_iter=n_iter,
            cv=cv,
            scoring=PRIMARY,
            n_jobs=-1,
            verbose=1,
            random_state=42,
            refit=True,
        )

    searcher.fit(X, y)

    # Evaluate the best pipeline with multiple metrics (no leakage: prep is inside)
    cv_res = cross_validate(
        searcher.best_estimator_, X, y, cv=cv, scoring=SCORERS, n_jobs=-1, return_train_score=False
    )
    row = {
        "Model": name,
        "BestParams": searcher.best_params_,
        **{f"mean_{k}": np.mean(v) for k, v in cv_res.items() if k.startswith("test_")},
        **{f"std_{k}": np.std(v) for k, v in cv_res.items() if k.startswith("test_")},
    }
    return searcher.best_estimator_, row


In [ ]:
rf = RandomForestClassifier(
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"  # helps with class imbalance
)

pipe_rf = Pipeline([
    ("prep", preprocessor),     # <-- your ColumnTransformer
    ("clf", rf)
])

# Parameter distributions for random search
param_dist = {
    "clf__n_estimators": randint(200, 801),         # 200–800
    "clf__max_depth": randint(5, 41),               # 5–40
    "clf__min_samples_split": randint(2, 21),       # 2–20
    "clf__min_samples_leaf": randint(1, 11),        # 1–10
    "clf__max_features": ["sqrt", "log2", 0.5, 0.7, None],
    "clf__bootstrap": [True, False]
}

rs = RandomizedSearchCV(
    estimator=pipe_rf,
    param_distributions=param_dist,
    n_iter=60,                # increase for more thorough search
    scoring=scorers,
    refit="f1",               # refit the best by F1
    cv=cv,
    verbose=1,
    n_jobs=-1,
    random_state=42,
    return_train_score=False
)

rs.fit(X_train, y_train)
print("Best (RandomizedSearch) F1:", rs.best_score_)
best_rand = rs.best_estimator_


In [ ]:
cvres = pd.DataFrame(rs.cv_results_)
cols = [
    "rank_test_f1",
    "mean_test_f1", "std_test_f1",
    "mean_test_accuracy", "mean_test_precision",
    "mean_test_recall", "mean_test_roc_auc"
]
param_cols = [c for c in cvres.columns if c.startswith("param_")]
leaderboard_rs = cvres[["mean_fit_time", *cols, *param_cols]].sort_values("rank_test_f1")
leaderboard_rs.head(10)


In [ ]:
def eval_on_holdout(pipe, X_test, y_test):
    proba = pipe.predict_proba(X_test)[:, 1]
    pred  = (proba >= 0.5).astype(int)
    return {
        "f1": f1_score(y_test, pred),
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred),
        "roc_auc": roc_auc_score(y_test, proba)
    }

best_rand_holdout = eval_on_holdout(best_rand, X_test, y_test)
best_grid_holdout = eval_on_holdout(best_grid, X_test, y_test)

pd.DataFrame(
    [best_rand_holdout, best_grid_holdout],
    index=["RandomizedSearch best", "GridSearch best"]
)
